# Hybrid Search Workshop - Keyword + Vector + RRF

**Goal (1 hour):** understand *why* production search combines keyword and vector retrieval, by building both from scratch and fusing them.

By the end you'll be able to explain:
- where **keyword search (BM25)** wins - exact terms, codes, IDs, names
- where **vector search (dense)** wins - synonyms, paraphrases, "meaning"
- how **RRF** fuses the two so you get the best of both

> we will fill in the `# TODO` cells live. Solutions are in `notebooks/solution.ipynb` if you get stuck.


## 0. Setup

Run this once. First run downloads a small (~80MB) embedding model.

In [ ]:
# If running locally for the first time, uncomment:
# !pip install -r ../requirements.txt

import sys, json, math, re
from collections import Counter, defaultdict
import numpy as np

print("Python", sys.version.split()[0])
print("numpy", np.__version__)

In [ ]:
# Load the mini knowledge base: 20 short internal-help docs
with open("../data/docs.json") as f:
    DOCS = json.load(f)

print(f"{len(DOCS)} docs loaded")
for d in DOCS[:3]:
    print(f"  [{d['id']}] {d['title']}")
ID2TITLE = {d["id"]: d["title"] for d in DOCS}

Notice the dataset has **paired** docs on purpose:
- `D01 VPN Access Request` vs `D02 Working Remotely` — same topic, different words
- `D07 Error Code E-4042` vs `D19 Error Code E-1001` — exact codes
- `D05 Expense Reimbursement` vs `D06 Getting Money Back for Purchases`

Keep an eye on these as we test queries.

## 1. one query, two failures

Run the cell below. We'll search the *naive* way twice and see neither is enough.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def naive_keyword(query, docs):
    """Count shared words. The dumbest possible keyword search."""
    q = set(tokenize(query))
    scored = []
    for d in docs:
        overlap = len(q & set(tokenize(d["title"] + " " + d["text"])))
        scored.append((d["id"], overlap))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [(i, s) for i, s in scored if s > 0][:3]

# A user asking in their own words:
q = "how do I connect to work systems from home"
print("Query:", q)
for i, s in naive_keyword(q, DOCS):
    print(f"  {s}  [{i}] {ID2TITLE[i]}")

Naive keyword matching leans on shared words. It'll likely surface the VPN doc because it shares words - but a doc that says *"remote access gateway"* and *"secure tunnel"* (the same answer, different words) gets **no credit**. That's the synonym problem. Let's do this properly.

## 2. BM25 - keyword search done right

BM25 improves on word-counting with two ideas:
1. **IDF** - rare words matter more than common ones ("E-4042" >> "the")
2. **Saturation** - the 10th occurrence of a word adds less than the 2nd

In [ ]:
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = docs, k1, b
        self.corpus = [tokenize(d["title"] + " " + d["text"]) for d in docs]
        self.doc_len = [len(c) for c in self.corpus]
        # average document length
        self.avgdl = sum(self.doc_len) / len(self.corpus)
        # total number of documents
        self.N = len(self.corpus)
        # term frequency
        self.tf = [Counter(c) for c in self.corpus]
        # document frequency
        self.df = defaultdict(int)
        for c in self.corpus:
            for term in set(c):
                self.df[term] += 1

        # print(self.df)
        # print(self.tf)

    def idf(self, term):
        """Inverse document frequency."""
        n = self.df.get(term, 0)
        # the IDF formula. no need to memorize this
        return math.log(1 + (self.N - n + 0.5) / (n + 0.5))

    def search(self, query, top_k=3):
        q = tokenize(query)
        scores = []
        for i in range(self.N):
            s = 0.0
            for term in q:
                if term not in self.tf[i]:
                    continue
                freq = self.tf[i][term]
                num = freq * (self.k1 + 1)
                den = freq + self.k1 * (
                    1 - self.b + self.b * self.doc_len[i] / self.avgdl
                )
                s += self.idf(term) * num / den
                num = freq * (self.k1 + 1)
                den = freq + self.k1 * (
                    1 - self.b + self.b * self.doc_len[i] / self.avgdl
                )
                s += self.idf(term) * num / den
            scores.append((self.docs[i]["id"], s))
        scores.sort(key=lambda x: x[1], reverse=True)
        return [(i, sc) for i, sc in scores if sc > 0][:top_k]


bm25 = BM25(DOCS)
print("BM25 ready")

[Counter({'the': 3, 'vpn': 2, 'access': 2, 'request': 2, 'to': 2, 'from': 2, 'your': 2, 'connect': 1, 'internal': 1, 'network': 1, 'home': 1, 'submit': 1, 'a': 1, 'through': 1, 'it': 1, 'portal': 1, 'approval': 1, 'manager': 1, 'is': 1, 'required': 1, 'once': 1, 'approved': 1, 'install': 1, 'globalprotect': 1, 'client': 1, 'and': 1, 'sign': 1, 'in': 1, 'with': 1, 'corporate': 1, 'credentials': 1}), Counter({'the': 2, 'to': 2, 'your': 2, 'working': 1, 'remotely': 1, 'employees': 1, 'who': 1, 'work': 1, 'from': 1, 'home': 1, 'can': 1, 'reach': 1, 'company': 1, 'systems': 1, 'securely': 1, 'using': 1, 'remote': 1, 'access': 1, 'gateway': 1, 'speak': 1, 'team': 1, 'lead': 1, 'get': 1, 'permission': 1, 'then': 1, 'set': 1, 'up': 1, 'secure': 1, 'tunnel': 1, 'software': 1, 'on': 1, 'laptop': 1}), Counter({'password': 2, 'reset': 2, 'you': 2, 'your': 2, 'if': 1, 'forget': 1, 'login': 1, 'visit': 1, 'the': 1, 'self': 1, 'service': 1, 'portal': 1, 'and': 1, 'click': 1, 'will': 1, 'receive': 1, 

In [ ]:
# BM25 shines on exact tokens. Try an error code:
for q in ["E-4042", "two-factor authentication", "VPN access"]:
    print("Q:", q)
    for i, s in bm25.search(q):
        print(f"   {s:5.2f}  [{i}] {ID2TITLE[i]}")
    print()

In [ ]:
# ...but watch it struggle when the user avoids the doc's exact words:
q = "I'm locked out and can't sign in"
print("Q:", q)
for i, s in bm25.search(q):
    print(f"   {s:5.2f}  [{i}] {ID2TITLE[i]}")
print(
    "\n(The Password Reset / Account Recovery docs barely match — they say"
    " 'reset', 'recovery', 'verification code', not 'locked out / sign in'.)"
)

## 3. Dense retrieval — search by meaning

Embeddings map text to vectors where **similar meaning → nearby vectors**, regardless of exact words. We use `all-MiniLM-L6-v2`, a small, fast, free model.

> **Offline fallback:** if the model can't download (no internet), the next cell
> falls back to a lightweight hashing embedding so the workshop still runs. The
> lesson is the same; the real model just separates meaning more cleanly.

In [7]:
USE_REAL_MODEL = True
embedder = None
if USE_REAL_MODEL:
    try:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer("all-MiniLM-L6-v2")
        def embed_texts(texts):
            return _model.encode(texts, normalize_embeddings=True)
        embedder = "all-MiniLM-L6-v2"
    except Exception as e:
        print("Real model unavailable, using fallback. Reason:", str(e)[:80])

# uncomment this to use the fallback
# if embedder is None:
#     DIM = 256
#     def _embed_one(text):
#         v = np.zeros(DIM)
#         toks = tokenize(text)
#         grams = toks + [a + "_" + b for a, b in zip(toks, toks[1:])]
#         for g in grams:
#             v[hash(g) % DIM] += 1.0
#         n = np.linalg.norm(v)
#         return v / n if n else v
#     def embed_texts(texts):
#         return np.vstack([_embed_one(t) for t in texts])
#     embedder = "hashing-fallback"

print("Embedder:", embedder)

Embedder: all-MiniLM-L6-v2


In [8]:
class Dense:
    def __init__(self, docs):
        self.docs = docs
        self.mat = embed_texts([d["title"] + " " + d["text"] for d in docs])

    def search(self, query, top_k=3):
        q = embed_texts([query])[0]
        sims = self.mat @ q  # cosine sim (vectors are normalized)
        order = np.argsort(-sims)[:top_k]
        return [(self.docs[i]["id"], float(sims[i])) for i in order]


dense = Dense(DOCS)

# The SAME locked-out query that stumped BM25:
q = "I'm locked out and can't sign in"
print("Q:", q)
for i, s in dense.search(q):
    print(f"   {s:5.2f}  [{i}] {ID2TITLE[i]}")
print("\nDense understands 'locked out' ~ 'account recovery' even with no shared words.")

Q: I'm locked out and can't sign in
    0.56  [D04] Account Recovery
    0.46  [D03] Password Reset
    0.44  [D19] Error Code E-1001

Dense understands 'locked out' ~ 'account recovery' even with no shared words.


In [9]:
# But dense has a blind spot: exact codes/IDs. Compare both on E-4042:
q = "E-4042"
print("Q:", q)
print(" BM25 :", [ID2TITLE[i] for i, _ in bm25.search(q)])
print(" Dense:", [ID2TITLE[i] for i, _ in dense.search(q)])
print("\nDense can muddle exact tokens; BM25 nails them. We want BOTH.")

Q: E-4042
 BM25 : ['Error Code E-4042', 'Error Code E-1001']
 Dense: ['Error Code E-4042', 'Error Code E-1001', 'Reporting a Phishing Email']

Dense can muddle exact tokens; BM25 nails them. We want BOTH.


## 4. RRF — fuse the two rankings

The two retrievers produce **incompatible score scales** (BM25 ~0–8, cosine ~0–1), so we can't just add scores. **Reciprocal Rank Fusion** fuses on *rank* instead:

$$\text{score}(d) = \sum_{\text{retrievers}} \frac{1}{k + \text{rank}(d)}, \quad k=60$$

You implement it.

In [ ]:
def rrf(result_lists, k=60, top_k=3):
    fused = defaultdict(float)
    for results in result_lists:
        for rank, (doc_id, _score) in enumerate(results):
            # reciprocal rank contribution
            fused[doc_id] += 1.0 / (k + rank + 1)
    out = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    return out[:top_k]

def hybrid_search(query, top_k=3):
    b = bm25.search(query, top_k=5)
    d = dense.search(query, top_k=5)
    return rrf([b, d], top_k=top_k)

print("Hybrid ready")

Hybrid ready


In [11]:
# The payoff: run all three side by side
def compare(query):
    print("=" * 60)
    print("Q:", query)
    rows = {
        "BM25 ": [ID2TITLE[i] for i, _ in bm25.search(query)],
        "Dense": [ID2TITLE[i] for i, _ in dense.search(query)],
        "HYBRID": [ID2TITLE[i] for i, _ in hybrid_search(query)],
    }
    for name, titles in rows.items():
        print(f"  {name}: {titles}")

for q in [
    "I'm locked out and can't sign in",   # dense rescues
    "E-1001",                              # bm25 rescues
    "how do I connect from home",          # paraphrase
    "get reimbursed for a work purchase",  # synonym-heavy
]:
    compare(q)

Q: I'm locked out and can't sign in
  BM25 : ['Error Code E-1001', 'Account Recovery', 'VPN Access Request']
  Dense: ['Account Recovery', 'Password Reset', 'Error Code E-1001']
  HYBRID: ['Account Recovery', 'Error Code E-1001', 'Password Reset']
Q: E-1001
  BM25 : ['Error Code E-1001', 'Error Code E-4042']
  Dense: ['Error Code E-1001', 'Error Code E-4042', 'Onboarding Checklist']
  HYBRID: ['Error Code E-1001', 'Error Code E-4042', 'Onboarding Checklist']
Q: how do I connect from home
  BM25 : ['VPN Access Request', 'Office WiFi', 'Working Remotely']
  Dense: ['VPN Access Request', 'Working Remotely', 'Office WiFi']
  HYBRID: ['VPN Access Request', 'Office WiFi', 'Working Remotely']
Q: get reimbursed for a work purchase
  BM25 : ['Getting Money Back for Purchases', 'Travel Booking', 'Working Remotely']
  Dense: ['Getting Money Back for Purchases', 'Expense Reimbursement Policy', 'Software Procurement']
  HYBRID: ['Getting Money Back for Purchases', 'Travel Booking', 'Expense Reimbur

**What you should see:** hybrid is never worse than the better of the two, and on mixed queries it's better than either alone. That's the whole pitch — you don't have to choose between exact and semantic.

## 5. (Optional) Put an LLM on top — RAG

Retrieval is 90% of RAG quality. Here we just feed the top hybrid docs to an LLM
to write the final answer. Uses **Groq's free tier** — get a key at
console.groq.com, then set it below.

In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""  # <-- paste your free key, or skip this section

def answer(query):
    hits = hybrid_search(query, top_k=3)
    context = "\n\n".join(
        f"[{i}] {ID2TITLE[i]}: {next(d['text'] for d in DOCS if d['id']==i)}"
        for i, _ in hits
    )
    try:
        from groq import Groq
        client = Groq()
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "Answer ONLY from the context. Cite doc IDs like [D03]. If not in context, say so."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
            ],
            temperature=0,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"(LLM skipped: {str(e)[:60]})\nTop docs were:\n{context[:400]}..."

print(answer("I forgot my password, what do I do?"))

## Recap

| Retriever | Wins on | Loses on |
|-----------|---------|----------|
| **BM25** (keyword) | exact terms, codes, IDs, names | synonyms, paraphrases |
| **Dense** (vector) | meaning, synonyms, fuzzy phrasing | exact tokens, rare codes |
| **RRF hybrid** | both | almost nothing — safe default |

**One-line takeaway:** *keyword finds exact, vector finds meaning, RRF gets you both.*

### Try it yourself
- Add a doc and a query that only hybrid gets right
- Change RRF `k` (try 1 vs 60) — how does it shift results?
- Swap in a different embedding model and re-run the comparisons
